In [209]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from xgboost import XGBRegressor
from sklearn import model_selection
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import lightgbm as lgb

In [210]:
data_file = Path('..') / 'model' / 'SUPERSTORE_DATASET_CLUSTERING.csv'
df = pd.read_csv(data_file, sep=',')

In [211]:
df.head()

,CUSTOMER_ID,ORDER_DATE,SHIP_MODE,CATEGORY,REGION,SEGMENT,NET_SALES,PROFIT,NET_SALES_SCALED,PROFIT_SCALED,CLUSTER
0,AA-10315,2019-03-31,STANDARD CLASS,OFFICE SUPPLIES,WEST,CONSUMER,204.7000,28.8658,-0.802781,-0.291871,1
1,AA-10375,2019-04-21,STANDARD CLASS,OFFICE SUPPLIES,WEST,CONSUMER,1171.1996,91.3899,-1.674506,-0.575691,1
2,AA-10480,2019-05-04,SAME DAY,FURNITURE,EAST,CONSUMER,357.4148,63.0594,-0.759041,-0.142015,0
3,AA-10645,2019-12-01,FIRST CLASS,FURNITURE,EAST,CONSUMER,291.2400,38.4204,-0.777994,-0.249997,0
4,AB-10015,2019-02-18,STANDARD CLASS,OFFICE SUPPLIES,CENTRAL,CONSUMER,59.1680,2.9553,-0.557327,-0.265966,1


In [ ]:
df_predict = df[['ORDER_DATE', 'SHIP_MODE', 'PROFIT_SCALED', 'NET_SALES_SCALED', 'REGION', 'CLUSTER']].copy()

In [ ]:
df_predict = pd.get_dummies(
    df_predict,
    columns=['REGION'])

In [214]:
df_predict['SHIP_MODE'].value_counts()

SHIP_MODE
STANDARD CLASS    445
SECOND CLASS      162
FIRST CLASS       119
SAME DAY           44
Name: count, dtype: int64

In [215]:
state_map = {
        'STANDARD CLASS': '0',
        'SECOND CLASS': '1',
        'FIRST CLASS': '2',
        'SAME DAY': '3'
}

df_predict['SHIP_MODE'] = df_predict['SHIP_MODE'].map(state_map)
df_predict['SHIP_MODE'] = df_predict['SHIP_MODE'].astype('Int64')

In [216]:
df_predict['ORDER_DATE'] = pd.to_datetime(df_predict['ORDER_DATE'], errors='coerce')
df_predict['ORDER_YEAR'] = df_predict['ORDER_DATE'].dt.year

df_predict = df_predict.drop(columns=['ORDER_DATE'])

In [217]:
df_predict.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 770 entries, 0 to 769
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   SHIP_MODE         770 non-null    Int64  
 1   PROFIT_SCALED     770 non-null    float64
 2   NET_SALES_SCALED  770 non-null    float64
 3   CLUSTER           770 non-null    int64  
 4   REGION_CENTRAL    770 non-null    bool   
 5   REGION_EAST       770 non-null    bool   
 6   REGION_SOUTH      770 non-null    bool   
 7   REGION_WEST       770 non-null    bool   
 8   ORDER_YEAR        770 non-null    int32  
dtypes: Int64(1), bool(4), float64(2), int32(1), int64(1)
memory usage: 31.0 KB


In [218]:
df_predict.head()

,SHIP_MODE,PROFIT_SCALED,NET_SALES_SCALED,CLUSTER,REGION_CENTRAL,REGION_EAST,REGION_SOUTH,REGION_WEST,ORDER_YEAR
0,0,-0.291871,-0.802781,1,False,False,False,True,2019
1,0,-0.575691,-1.674506,1,False,False,False,True,2019
2,3,-0.142015,-0.759041,0,False,True,False,False,2019
3,2,-0.249997,-0.777994,0,False,True,False,False,2019
4,0,-0.265966,-0.557327,1,True,False,False,False,2019


In [219]:
dfs_clusters = {
    cluster: df_predict[df_predict['CLUSTER'] == cluster]
    for cluster in df_predict['CLUSTER'].unique()
}

In [220]:
X_train = []
X_test = []
y_train = []
y_test = []

for i in range(0, 3):

    X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
        dfs_clusters[i].drop('PROFIT_SCALED', axis=1),
        dfs_clusters[i]['PROFIT_SCALED'],
        test_size=0.2,
        random_state=0
    )

    X_train.append(X_train_i)
    X_test.append(X_test_i)
    y_train.append(y_train_i)
    y_test.append(y_test_i)
    

In [221]:
models = {
    'XGBOOST': XGBRegressor(random_state=0),
    'REGRESSAO LINEAR': LinearRegression(),
    'RANDOM FOREST': RandomForestRegressor(random_state=0),
    'SVR': SVR(),
    'LIGHTGBM': lgb.LGBMRegressor(
        random_state=0,
        force_col_wise=True,
        verbose=-1,
        n_jobs=1
    )
}

for d in [2, 3, 4]:
    models[f'REGRESSAO POLINOMIAL GRAU {d}'] = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('model', LinearRegression())
    ])

kfold = model_selection.KFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

print('=' * 40)
print('RESULTADOS DA EXPLORACAO (REGRESSAO)')
print('=' * 40, '\n')

results = []

for i in range(0, 3):
    results = []
    for name, model in models.items():
        scores = model_selection.cross_validate(
            model,
            X_train[i],
            y_train[i],
            scoring=['neg_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
            cv=kfold,
            n_jobs=1
        )

        mse = round(-scores['test_neg_mean_squared_error'].mean(), 4)
        mae = round(-scores['test_neg_mean_absolute_error'].mean(), 4)
        r2 = round(scores['test_r2'].mean(), 4)

        result = {
            'NAME': name,
            'MSE': mse,
            'MAE': mae,
            'R2': r2
        }
        results.append(result)

    print('CLUSTER ', i)
    results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
    display(results_df)



RESULTADOS DA EXPLORACAO (REGRESSAO)

CLUSTER  0


,NAME,MSE,MAE,R2
0,REGRESSAO LINEAR,0.0098,0.0788,0.6090
1,LIGHTGBM,0.0118,0.0793,0.5955
2,REGRESSAO POLINOMIAL GRAU 2,0.0110,0.0807,0.5606
3,RANDOM FOREST,0.0108,0.0791,0.5573
4,XGBOOST,0.0137,0.0909,0.4497
5,REGRESSAO POLINOMIAL GRAU 3,0.0170,0.0929,0.3709
6,REGRESSAO POLINOMIAL GRAU 4,0.0270,0.1060,-0.1282
7,SVR,0.0345,0.1390,-0.2520


CLUSTER  1


,NAME,MSE,MAE,R2
0,REGRESSAO POLINOMIAL GRAU 2,0.0080,0.0695,0.7050
1,REGRESSAO LINEAR,0.0079,0.0692,0.7028
2,LIGHTGBM,0.0095,0.0758,0.6602
3,RANDOM FOREST,0.0100,0.0773,0.6338
4,REGRESSAO POLINOMIAL GRAU 3,0.0100,0.0738,0.6320
5,XGBOOST,0.0130,0.0875,0.5240
6,REGRESSAO POLINOMIAL GRAU 4,0.0138,0.0771,0.4907
7,SVR,0.0336,0.1414,-0.0884


CLUSTER  2


,NAME,MSE,MAE,R2
0,REGRESSAO LINEAR,0.0085,0.0714,0.6202
1,REGRESSAO POLINOMIAL GRAU 2,0.0089,0.0735,0.6042
2,LIGHTGBM,0.0091,0.0730,0.5816
3,RANDOM FOREST,0.0095,0.0743,0.5632
4,REGRESSAO POLINOMIAL GRAU 3,0.0102,0.0770,0.5446
5,XGBOOST,0.0120,0.0846,0.4549
6,REGRESSAO POLINOMIAL GRAU 4,0.0155,0.0841,0.2541
7,SVR,0.0234,0.1185,-0.0269


Temos em todos os modelos que REGRESSÃO LINEAR é o modelo preditivo mais promissor.

In [222]:
regularization_models = {
    'RIDGE': Ridge(max_iter=5000),
    'LASSO': Lasso(max_iter=5000),
    'ELASTICNET': ElasticNet(max_iter=5000, tol=1e-3)
}

param_grids = {
    'RIDGE': {
        'alpha': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    'LASSO': {
        'alpha': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    'ELASTICNET': {
        'alpha': [0.001, 0.01, 0.1, 1, 10],
        'l1_ratio': [0.35, 0.45, 0.55, 0.65]
    }
}

results_tuning = []

for i in range(0, 3):
    for model_name, model in regularization_models.items():
        param_grid = param_grids[model_name]

        grid_search = model_selection.GridSearchCV(
            model,
            param_grid,
            scoring='r2',
            cv=kfold,
            n_jobs=-1,
            verbose=0
        )

        grid_search.fit(X_train[i], y_train[i])
        best_estimator = grid_search.best_estimator_

        r2_cruzada = round(grid_search.best_score_, 4)
        r2_teste = round(best_estimator.score(X_test[i], y_test[i]), 4)

        result = {
            'CLUSTER': i,
            'MODEL': model_name,
            'BEST_PARAMS': grid_search.best_params_,
            'CV_R2': r2_cruzada,
            'TEST_R2': r2_teste,
            'R2_GAP': round(abs(r2_teste - r2_cruzada), 4),
            'ESTIMATOR': best_estimator
        }
        results_tuning.append(result)

results_tuning_df = pd.DataFrame(results_tuning)
results_tuning_view = results_tuning_df.drop(columns=['ESTIMATOR']).sort_values(
    ['CLUSTER', 'TEST_R2'], ascending=[True, False]
).reset_index(drop=True)

display(results_tuning_view)



,CLUSTER,MODEL,BEST_PARAMS,CV_R2,TEST_R2,R2_GAP
0,0,LASSO,{'alpha': 0.01},0.6232,0.6572,0.0340
1,0,ELASTICNET,"{'alpha': 0.01, 'l1_ratio': 0.65}",0.6271,0.6562,0.0291
2,0,RIDGE,{'alpha': 1},0.6135,0.6322,0.0187
3,1,ELASTICNET,"{'alpha': 0.01, 'l1_ratio': 0.35}",0.7146,0.5606,0.1540
4,1,LASSO,{'alpha': 0.01},0.7092,0.5597,0.1495
5,1,RIDGE,{'alpha': 1},0.7064,0.5567,0.1497
6,2,RIDGE,{'alpha': 1},0.6206,0.6655,0.0449
7,2,LASSO,{'alpha': 0.001},0.6245,0.6652,0.0407
8,2,ELASTICNET,"{'alpha': 0.01, 'l1_ratio': 0.35}",0.6266,0.6463,0.0197


In [223]:
best_models_df = (
    results_tuning_df
    .sort_values(['CLUSTER', 'TEST_R2'], ascending=[True, False])
    .groupby('CLUSTER', as_index=False)
    .first()
    .reset_index(drop=True)
    .rename(
        columns={
            'MODEL': 'BEST_MODEL',
            'BEST_PARAMS': 'BEST_PARAMS',
            'CV_R2': 'BEST_CV_R2',
            'TEST_R2': 'BEST_TEST_R2',
            'R2_GAP': 'BEST_R2_GAP'
        }
    )
)

best_models_view = best_models_df[[
    'CLUSTER', 'BEST_MODEL', 'BEST_PARAMS', 'BEST_CV_R2', 'BEST_TEST_R2', 'BEST_R2_GAP'
]]

display(best_models_view)

model_dir = Path('..') / 'model'
model_dir.mkdir(parents=True, exist_ok=True)

best_models_path = model_dir / 'BEST_MODELS_BY_CLUSTER.csv'
best_models_view.to_csv(best_models_path, index=False)

,CLUSTER,BEST_MODEL,BEST_PARAMS,BEST_CV_R2,BEST_TEST_R2,BEST_R2_GAP
0,0,LASSO,{'alpha': 0.01},0.6232,0.6572,0.0340
1,1,ELASTICNET,"{'alpha': 0.01, 'l1_ratio': 0.35}",0.7146,0.5606,0.1540
2,2,RIDGE,{'alpha': 1},0.6206,0.6655,0.0449


Arquivo salvo em: ../model/BEST_MODELS_BY_CLUSTER.csv


Exception ignored in: <function ResourceTracker.__del__ at 0x7f06ac66cae0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7fa608c70ae0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7fe496868ae0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/reso